# Assignment 07 - Notebook 03
## Diabetes Knowledge Agent và triển khai

Agent phối hợp bộ truy xuất BM25 với Knowledge Graph, trả lời có nguồn và nêu giới hạn sử dụng.

## 1. Khởi tạo

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)

Project: C:\DATA\assignment3\assignment7


In [2]:
from src.rag_agent import DiabetesAgent
agent = DiabetesAgent(ROOT/'knowledge')
print('Tài liệu:', len(agent.documents), '| Triple:', len(agent.triples))

Tài liệu: 6 | Triple: 14


## 2. Luồng Agent

1) nhận câu hỏi; 2) chọn công cụ BM25; 3) mở rộng entity bằng graph; 4) tổng hợp bằng chứng; 5) trả lời kèm nguồn. Agent không tự đặt chẩn đoán.

In [3]:
result = agent.answer('Đái tháo đường có thể gây biến chứng gì?', top_k=3)
print(result['answer'])
print('Nguồn:', [x['title'] for x in result['sources']])

Đái tháo đường là bệnh mạn tính xảy ra khi tuyến tụy không sản xuất đủ insulin hoặc cơ thể không sử dụng hiệu quả insulin. Tăng đường huyết kéo dài có thể gây tổn thương tim, mạch máu, mắt, thận và thần kinh. Type 1 liên quan thiếu insulin và cần dùng insulin; type 2 thường liên quan tình trạng cơ thể sử dụng insulin không hiệu quả. Các xét nghiệm thường dùng để đánh giá đái tháo đường gồm HbA1c, glucose huyết tương lúc đói, nghiệm pháp dung nạp glucose đường uống và glucose huyết tương ngẫu nhiên khi có triệu chứng. Bác sĩ cần xác nhận và diễn giải kết quả trong bối cảnh lâm sàng. Các yếu tố liên quan nguy cơ đái tháo đường type 2 gồm thừa cân hoặc béo phì, ít vận động, tiền sử gia đình, tuổi cao hơn, tăng huyết áp và tiền đái tháo đường. Nguy cơ thống kê không phải chẩn đoán cá nhân.

Liên kết trong Knowledge Graph: đái tháo đường -có loại-> type 1; đái tháo đường -có loại-> type 2; đái tháo đường -được đánh giá bằng-> HbA1c; đái tháo đường -được đánh giá bằng-> glucose huyết tương.


## 3. Ví dụ câu hỏi semantic và multi-hop

In [4]:
questions = [
 'Dấu hiệu nào thường gặp?',
 'Xét nghiệm nào dùng để đánh giá?',
 'Type 2 liên quan yếu tố nào và có thể cải thiện ra sao?'
]
for q in questions:
    r = agent.answer(q)
    print('\nQ:', q, '\nA:', r['answer'][:320])


Q: Dấu hiệu nào thường gặp? 
A: Dấu hiệu có thể gặp gồm tiểu nhiều, khát nhiều, đói nhiều, sụt cân không chủ ý, mệt mỏi, nhìn mờ, vết thương chậm lành và tê bì tay chân. Một số người type 2 có ít hoặc không có triệu chứng rõ ràng. Đái tháo đường là bệnh mạn tính xảy ra khi tuyến tụy không sản xuất đủ insulin hoặc cơ thể không sử dụng hiệu quả insulin

Q: Xét nghiệm nào dùng để đánh giá? 
A: Các xét nghiệm thường dùng để đánh giá đái tháo đường gồm HbA1c, glucose huyết tương lúc đói, nghiệm pháp dung nạp glucose đường uống và glucose huyết tương ngẫu nhiên khi có triệu chứng. Bác sĩ cần xác nhận và diễn giải kết quả trong bối cảnh lâm sàng. Mô hình học máy của bài tập ước lượng nhãn từ dữ liệu quan sát lịc

Q: Type 2 liên quan yếu tố nào và có thể cải thiện ra sao? 
A: Các yếu tố liên quan nguy cơ đái tháo đường type 2 gồm thừa cân hoặc béo phì, ít vận động, tiền sử gia đình, tuổi cao hơn, tăng huyết áp và tiền đái tháo đường. Nguy cơ thống kê không phải chẩn đoán cá nhân. Đái tháo đườ

## 4. API Flask

Web cung cấp `POST /api/chat` với JSON `{question, top_k}` và `GET /health`. Render chạy Gunicorn theo `render.yaml`.

In [5]:
print((ROOT/'render.yaml').read_text(encoding='utf-8'))

services:
  - type: web
    name: assignment7-knowledge-rag-agent
    runtime: python
    plan: free
    buildCommand: pip install -r requirements.txt
    startCommand: gunicorn --chdir web app:app --workers 1 --threads 4
    healthCheckPath: /health



## 5. Công cụ dự đoán của Agent

Checkpoint MLP PyTorch tốt nhất từ Assignment 3 được xuất sang NumPy. Preprocessor giữ đúng 15 cột gốc; web nhận các chỉ số chính và điền giá trị mặc định cho trường còn lại. Điểm chưa calibration nên không được gọi là xác suất mắc bệnh.

In [6]:
sys.path.insert(0, str(ROOT/'web'))
from app import predict_risk
predict_risk({'age':45,'bmi':27.5,'hbA1c_level':5.7,'blood_glucose_level':120,'hypertension':0,'heart_disease':0})

{'class': 0, 'score': 0.1555, 'model': 'PyTorch MLP baseline_lr001 (NumPy inference)', 'notice': 'Điểm mô hình chưa hiệu chỉnh và không phải xác suất chẩn đoán y khoa.'}

## 6. Giới hạn và đạo đức

Corpus chỉ có sáu đoạn nên độ bao phủ thấp; BM25 nhạy với từ khóa; graph được xây thủ công. Hệ thống cần chuyên gia y tế thẩm định trước mọi ứng dụng thực tế. Đầu ra chỉ phục vụ học tập, không thay thế chẩn đoán, kê đơn hoặc điều trị.